## 14_ Model Registry & Governance
Explainable AI Credit Risk Decision Platform: Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Reason
To package the final optimised XGBoost model, preprocessing artefacts, metadata, and feature information into a single reproducible registry for governance, auditability, and deployment.

In [49]:
# IMPORTS & CONFIGURATION
# ____________________________________________________

from __future__ import annotations

import shutil
import warnings
from datetime import datetime
import logging

import joblib
import numpy as np

from src.utils.logger import get_logger
from src.utils.helpers import section
from src.utils.helpers import clear_memory
from src.utils.helpers import json

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
FEATURE_STORE_DIR = PROJECT_ROOT / "data" / "feature_store"

MODEL_REGISTRY_DIR = MODEL_DIR / "registry"

MODEL_REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = "14_model_registry"

MODEL_NAME = "Optimised_XGBoost"

MODEL_VERSION = "1.0.0"

TARGET = "loan_status"

print("MODEL REGISTRY & GOVERNANCE")

print(f"Notebook : {NOTEBOOK_NAME}")
print(f"Model    : {MODEL_NAME}")
print(f"Version  : {MODEL_VERSION}")
print("="*50)

MODEL REGISTRY & GOVERNANCE
Notebook : 14_model_registry
Model    : Optimised_XGBoost
Version  : 1.0.0


In [59]:
# REUSABLE REGISTRY UTILITIES
# ___________________________________________

def section(title: str):

    print(title)

    print("=" * 50)

# Logger
# _____________________

logger = get_logger()

# compute
# ________________________________________

def compute_sha256(filepath: Path):

    sha = hashlib.sha256()

    with open(filepath, "rb") as file:

        while True:

            chunk = file.read(4096)

            if not chunk:

                break

            sha.update(chunk)

    return sha.hexdigest()
    
# clear memory
#___________________________________________
clear_memory()

# save json
# ___________________________________________


def save_json(data, filepath: Path):

    with open(filepath, "w") as file:

        json.dump(data, file, indent=4)

# Load file
# ____________________________________________

def load_model(filename):

    logger.info(f"Loading model: {filename}")

    model = joblib.load(MODEL_DIR / filename)

    logger.info("Model loaded successfully.")

    return model
    
# exporting dataframe
# ____________________________________________
def export_dataframe(df: pd.DataFrame, filepath: Path):

    df.to_csv(filepath, index=False)


In [58]:
# LOAD CHAMPION MODEL
# __________________________________________________________

print("Load Champion Model")

champion_model = load_model("champion_xgboost.pkl")

logger.info(type(champion_model))

2026-07-23 11:11:02,655 | INFO | Loading model: champion_xgboost.pkl
2026-07-23 11:11:02,695 | INFO | Model loaded successfully.
2026-07-23 11:11:02,702 | INFO | <class 'xgboost.sklearn.XGBClassifier'>


Load Champion Model


In [20]:
# LOAD MODEL PERFORMANCE
# __________________________________________________

section("Load Model Performance")

performance = pd.read_csv(

    REPORT_DIR /

    "model_comparison.csv")

display(performance)

champion_results = (

    performance

    .loc[performance["Model"]=="Optimised XGBoost"]

    .iloc[0])

print(champion_results)

2026-07-22 10:49:00,632 | INFO | Load Model Performance
2026-07-22 10:49:00,634 | INFO | ==================================================


,Model,Dataset,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline XGBoost,Matrix D,0.63508,0.266932,0.704602,0.387183,0.724106
1,Optimised XGBoost,Matrix D,0.63182,0.266072,0.711081,0.387245,0.724671


Model        Optimised XGBoost
Dataset               Matrix D
Accuracy               0.63182
Precision             0.266072
Recall                0.711081
F1                    0.387245
ROC_AUC               0.724671
Name: 1, dtype: object


In [22]:
# BUILD MODEL METADATA
# ___________________________________________________________

section("Create Model Metadata")

logger.info("Loading feature metadata...")

# Load Matrix D Feature Names
# __________________________________________

feature_names = pd.read_csv(FEATURE_STORE_DIR /"matrix_D_feature_names.csv")

n_features = len(feature_names)

logger.info(f"Detected {n_features} features.")

# Build Metadata
# _______________________________________ 

metadata = {

    "model_name": MODEL_NAME,

    "version": MODEL_VERSION,

    "algorithm": "Optimised XGBoost",

    "feature_matrix": "Matrix D",

    "target": TARGET,

    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "random_state": RANDOM_STATE,

    "number_of_features": n_features,

    "feature_file": "matrix_D_feature_names.csv",

    "performance":{

        "accuracy": float(champion_results["Accuracy"]),

        "precision": float(champion_results["Precision"]),

        "recall": float(champion_results["Recall"]),

        "f1": float(champion_results["F1"]),

        "roc_auc": float(champion_results["ROC_AUC"])}}

metadata_path = save_json(metadata,"model_metadata.json")

logger.info("Model metadata created.")

display(pd.DataFrame(metadata.items(), columns=["Field","Value"]))

2026-07-22 11:04:28,728 | INFO | Create Model Metadata
2026-07-22 11:04:28,740 | INFO | ==================================================
2026-07-22 11:04:28,741 | INFO | Loading feature metadata...
2026-07-22 11:04:28,774 | INFO | Detected 91 features.
2026-07-22 11:04:28,777 | INFO | Model metadata created.


,Field,Value
0,model_name,Optimised_XGBoost
1,version,1.0.0
2,algorithm,Optimised XGBoost
3,feature_matrix,Matrix D
4,target,loan_status
5,training_date,2026-07-22 11:04:28
6,random_state,42
7,number_of_features,91
8,feature_file,matrix_D_feature_names.csv
9,performance,"{'accuracy': 0.63182, 'precision': 0.266072041..."


In [30]:
# REGISTER CHAMPION MODEL
# ______________________________________________________

section("Register Champion Model")

logger.info("Registering champion model...")

model_path = MODEL_REGISTRY_DIR / "champion_xgboost.pkl"

joblib.dump(champion_model,model_path)

checksum = compute_sha256(model_path)

save_json({"model_file":"champion_xgboost.pkl",
           
           "sha256":checksum},
          
          "model_checksum.json")

logger.info("Champion model registered.")

print(model_path)

2026-07-22 12:07:58,639 | INFO | Register Champion Model
2026-07-22 12:07:58,640 | INFO | ==================================================
2026-07-22 12:07:58,641 | INFO | Registering champion model...
2026-07-22 12:07:58,653 | INFO | Champion model registered.


/Users/emmanuelahadzi/models/registry/champion_xgboost.pkl


In [32]:
# PACKAGE DEPLOYMENT ARTIFACTS
# ______________________________________________________

section("Package Deployment Artifacts")

artifacts = [(MODEL_DIR / "optimised_xgboost.pkl",
              
              MODEL_REGISTRY_DIR / "champion_xgboost.pkl"),
             
             (FEATURE_STORE_DIR / "matrix_D_feature_names.csv",
              
              MODEL_REGISTRY_DIR / "matrix_D_feature_names.csv")]

for source, destination in artifacts:

    if source.exists():

        shutil.copy2(source, destination)

        logger.info(f"Copied {source.name}")

    else:

        logger.warning(f"{source.name} not found")

2026-07-22 12:23:57,271 | INFO | Package Deployment Artifacts
2026-07-22 12:23:57,272 | INFO | ==================================================
2026-07-22 12:23:57,279 | INFO | Copied optimised_xgboost.pkl
2026-07-22 12:23:57,282 | INFO | Copied matrix_D_feature_names.csv


In [33]:
# Search for preprocessing objects
# ____________________________________________

possible_files = [

    "structured_preprocessor.pkl",

    "tfidf_vectorizer.pkl",

    "label_encoder.pkl",

    "target_encoder.pkl"]

for filename in possible_files:

    source = MODEL_DIR / filename

    if source.exists():

        shutil.copy2(

            source,

            MODEL_REGISTRY_DIR / filename)

        logger.info(f"Copied {filename}")

2026-07-22 12:25:16,377 | INFO | Copied structured_preprocessor.pkl


In [34]:
# REGISTRY MANIFEST
# ________________________________________________

manifest = {

    "model":"champion_xgboost.pkl",

    "feature_names":"matrix_D_feature_names.csv",

    "metadata":"model_metadata.json",

    "checksum":"model_checksum.json",

    "model_card":"model_card.md",

    "registry_version":"1.0.0",

    "created":datetime.now().isoformat()}

save_json(manifest,"registry_manifest.json")

logger.info("Registry manifest created.")

2026-07-22 12:25:59,854 | INFO | Registry manifest created.


In [37]:
# CREATE PIPELINE CONFIGURATION
# _________________________________________________

section("Create Pipeline Configuration")

pipeline_config = {

    "project_name": "Explainable AI Credit Risk Decision Platform",

    "model_name": MODEL_NAME,

    "model_version": MODEL_VERSION,

    "algorithm": "Optimised XGBoost",

    "feature_matrix": "Matrix D",

    "target_column": TARGET,

    "random_state": RANDOM_STATE,

    "feature_names_file": "matrix_D_feature_names.csv",

    "model_file": "champion_xgboost.pkl",

    "metadata_file": "model_metadata.json",

    "checksum_file": "model_checksum.json",

    "model_card_file": "model_card.md",

    "preprocessors":{

        "structured_preprocessor":"structured_preprocessor.pkl",

        "tfidf_vectorizer":"tfidf_vectorizer.pkl"},

    "created":datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

save_json(pipeline_config,"pipeline_config.json")

logger.info("Pipeline configuration saved successfully.")

2026-07-22 13:01:02,017 | INFO | Create Pipeline Configuration
2026-07-22 13:01:02,018 | INFO | ==================================================
2026-07-22 13:01:02,029 | INFO | Pipeline configuration saved successfully.


In [35]:
# REGISTRY INVENTORY
# ____________________________________________________

inventory = []

for file in sorted(MODEL_REGISTRY_DIR.glob("*")):

    inventory.append({

        "File":file.name,

        "Size (KB)":round(

            file.stat().st_size/1024,2),

        "Type":file.suffix})

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

export_dataframe(inventory_df,"registry_inventory.csv")

,File,Size (KB),Type
0,champion_xgboost.pkl,195.93,.pkl
1,matrix_D_feature_names.csv,1.32,.csv
2,model_card.md,0.40,.md
3,model_checksum.json,0.12,.json
4,model_metadata.json,0.51,.json
5,registry_manifest.json,0.27,.json
6,registry_summary.csv,0.18,.csv
7,structured_preprocessor.pkl,9.53,.pkl


PosixPath('/Users/emmanuelahadzi/models/registry/registry_inventory.csv')

In [38]:
# FINAL REGISTRY VALIDATION
# ________________________________________________

section("Registry Validation")

required = [

    "champion_xgboost.pkl",

    "matrix_D_feature_names.csv",

    "model_metadata.json",

    "model_checksum.json",

    "model_card.md",

    "registry_manifest.json",

    "pipeline_config.json"]

missing = []

for file in required:

    if not (MODEL_REGISTRY_DIR /file
           
           ).exists():missing.append(file)

if len(missing)==0:

    logger.info("Registry validation passed.")

else:

    logger.warning("Missing files:")

    logger.warning(missing)

2026-07-22 13:01:28,692 | INFO | Registry Validation
2026-07-22 13:01:28,695 | INFO | ==================================================
2026-07-22 13:01:28,698 | INFO | Registry validation passed.


In [39]:
# CREATE MODEL CARD
# _________________________________________________

section("Create Model Card")

model_card = f"""
# Model Card

## Model

Optimised XGBoost

## Version

{MODEL_VERSION}

## Dataset

Matrix D

## Target

{TARGET}

## Features

{n_features}

## Performance

Accuracy : {champion_results['Accuracy']:.4f}

Precision : {champion_results['Precision']:.4f}

Recall : {champion_results['Recall']:.4f}

F1 Score : {champion_results['F1']:.4f}

ROC-AUC : {champion_results['ROC_AUC']:.4f}

## Explainability

SHAP TreeExplainer

## Feature Sources

• Structured Borrower Variables

• TF-IDF Loan Text

• FRED Macroeconomic Indicators

• GDELT News Sentiment

"""

with open(MODEL_REGISTRY_DIR /"model_card.md","w"
         
         ) as f:f.write(model_card)

logger.info("Model card generated.")

2026-07-22 13:01:32,478 | INFO | Create Model Card
2026-07-22 13:01:32,480 | INFO | ==================================================
2026-07-22 13:01:32,484 | INFO | Model card generated.


In [40]:
# GOVERNANCE VALIDATION
# _____________________________________________

section("Governance Validation")

required_files = [

    "champion_xgboost.pkl",

    "model_metadata.json",

    "model_checksum.json",

    "model_card.md"]

validation = []

for file in required_files:

    exists = (MODEL_REGISTRY_DIR /file
             
             ).exists()

    validation.append({"File":file,
                       
                       "Available":exists})

validation_df = pd.DataFrame(validation)

display(validation_df)

assert validation_df["Available"].all()

logger.info("Governance validation completed successfully.")

2026-07-22 13:01:35,513 | INFO | Governance Validation
2026-07-22 13:01:35,514 | INFO | ==================================================


,File,Available
0,champion_xgboost.pkl,True
1,model_metadata.json,True
2,model_checksum.json,True
3,model_card.md,True


2026-07-22 13:01:35,534 | INFO | Governance validation completed successfully.


In [46]:
# EXECUTIVE SUMMARY
# ____________________________________________

section("Executive Summary")

summary = pd.DataFrame({

    "Item":[

        "Champion Model",

        "Algorithm",

        "Feature Matrix",

        "Number of Features",

        "ROC-AUC",

        "Registry Location"],

    "Value":[

        MODEL_NAME,

        "Optimised XGBoost",

        "Matrix D",n_features,

        round(champion_results["ROC_AUC"],4),

        str(MODEL_REGISTRY_DIR)]})

display(summary)

export_dataframe(summary,"registry_summary.csv")

logger.info("Model registry completed successfully.")

print("\n Notebook 14 completed successfully.")

2026-07-23 10:32:45,807 | INFO | Executive Summary
2026-07-23 10:32:45,810 | INFO | ==================================================


,Item,Value
0,Champion Model,Optimised_XGBoost
1,Algorithm,Optimised XGBoost
2,Feature Matrix,Matrix D
3,Number of Features,91
4,ROC-AUC,0.7247
5,Registry Location,/Users/emmanuelahadzi/models/registry


2026-07-23 10:32:45,842 | INFO | Model registry completed successfully.



 Notebook 14 completed successfully.
